# Setup: Genie Data + Normalized TablesThis notebook decomposes the synthetic supply chain dataset into normalized Unity Catalog tables (`nodes`, `edges`, `bom`) that back the **Genie space** and the **optimization MCP server**, replacing the single `dataset_small.json` blob that the original single-agent read.It is the first step of the multi-agent (Genie + MCP + Supervisor) flow. Run it before `02b_evaluate_supervisor`.

## Cluster ConfigurationTested on Databricks Runtime 17.3 LTS ML. Single node is sufficient.

In [ ]:
%pip install -r ../requirements.txt --quietdbutils.library.restartPython()

## ConfigurationReuse the same catalog/schema/volume names as the rest of the accelerator.

In [ ]:
catalog = "supply_chain_stress_test"  # Change hereschema = "data"                       # Change herevolume = "operational"                # Change here_ = spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalog}')_ = spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}')_ = spark.sql(f'CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}')

## Generate and decompose the dataset`generate_data` produces the nested dict. `explode_dataset_to_tables` splits it into three DataFrames. `reconstruct_dataset_from_frames` is the exact inverse the MCP resolver uses — we assert parity so the optimizer sees identical input.

In [ ]:
import scripts.utils as utilsimport scripts.dataset_io as diodataset = utils.generate_data(N1=5, N2=10, N3=20)  # DO NOT CHANGEtables = dio.explode_dataset_to_tables(dataset)nodes_pd, edges_pd, bom_pd = tables['nodes'], tables['edges'], tables['bom']print('nodes', nodes_pd.shape, '| edges', edges_pd.shape, '| bom', bom_pd.shape)

In [ ]:
# Parity guard: reconstruct and compare against the original dictroundtrip = dio.reconstruct_dataset_from_frames(nodes_pd, edges_pd, bom_pd)assert dio.datasets_equivalent(dataset, roundtrip), 'reconstruction mismatch!'print('Reconstruction parity: OK')

## Write the normalized Delta tablesManaged tables with descriptive comments — Genie relies on the comments for natural-language accuracy.

In [ ]:
for name, pdf in [('nodes', nodes_pd), ('edges', edges_pd), ('bom', bom_pd)]:    sdf = spark.createDataFrame(pdf.astype(object).where(pdf.notna(), None))    sdf.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{catalog}.{schema}.{name}')    print('wrote', name, spark.table(f'{catalog}.{schema}.{name}').count(), 'rows')

In [ ]:
C = f'{catalog}.{schema}'comments = [  (f"COMMENT ON TABLE {C}.nodes IS 'One row per node in the 3-tier supply chain network. tier 1 = finished-goods products, tier 2 = direct suppliers, tier 3 = sub-suppliers. inventory and capacity are defined for every node; profit_margin and demand apply only to tier-1 products; material_type is the material a supplier produces and is null for tier-1 products.'"),  (f"COMMENT ON TABLE {C}.edges IS 'Directed supply links: material flows from source_node (upstream supplier) to target_node (downstream consumer). To find downstream production sites for a supplier, filter source_node and read target_node.'"),  (f"COMMENT ON TABLE {C}.bom IS 'Bill of materials: which material types each product/assembly node requires, with quantity_required per unit.'"),]for c in comments:    spark.sql(c)print('table comments applied (see agent/00 notes for full column-level comments)')

## Backward compatibilityKeep writing `dataset_small.json` so the existing notebooks (`02_stress_testing`, `03_deploy_agent`) and the original single-agent still work.

In [ ]:
import jsonwith open(f'/Volumes/{catalog}/{schema}/{volume}/dataset_small.json', 'w') as fh:    json.dump(dataset, fh)print('wrote dataset_small.json')

## Next steps1. Create a **Genie space** over `nodes`, `edges`, `bom` (see `agent/README.md`).2. Deploy the **optimization MCP server** in `mcp_server/` to Databricks Apps.3. Wire both into a **Multi-Agent Supervisor**.4. Evaluate with `agent/02b_evaluate_supervisor`.